# 06 — GRU full: hoàn thiện 3 seed

Giữ seed **1234** đã chạy ở notebook 05; chỉ train thêm **317, 2026**,
mỗi seed có `none` và `balanced`, mỗi loss 8 epoch.
Tổng hợp 3 seed: 6 model, 48 epoch, 12 file predictions.
Phần chạy mới: 4 model, 32 epoch, 8 predictions; khoảng **45 phút fit**
nếu tốc độ tương tự lượt RTX 3050 đã đo. Đây là dự trù, không phải số đo mới.

Split random 64/16/20 cố định random_state=42; seed ở đây thay đổi khởi tạo
và minibatch, không thay đổi split. Preprocessing outer80% giữ như code tác giả.
Mọi split gồm tháng 7; không ghép với temporal LightGBM của notebook 03.
Không chọn seed hoặc cấu hình theo random test.


In [1]:
from pathlib import Path
import json
import platform
import sys

PROJECT_DIR = None  # Hoặc r"D:\UIT\DS317\BAF_project"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next((p for p in [here, *here.parents] if (p / "src/baf.py").is_file()), None)
)
if ROOT is None:
    raise FileNotFoundError("Mở BAF_project hoặc điền PROJECT_DIR.")
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf
import gru

for module in [baf, gru]:
    baf.require(
        Path(module.__file__).resolve()
        == (ROOT / "src" / f"{module.__name__}.py").resolve(),
        "Kernel đang giữ source khác. Restart Kernel rồi Run All.",
    )
baf.require(
    hasattr(baf, "summarize_operating_points"),
    "Cần source của gói bước 06. Restart Kernel.",
)
print("Kernel:", sys.executable)
print("Torch:", gru.torch.__version__)
print("CUDA:", gru.torch.cuda.is_available())


Kernel: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Torch: 2.14.1+cu130
CUDA: True


## 1. Đọc lại seed 1234 và kiểm tra trước khi train

Mặc định dùng đúng thư mục run đã hoàn tất ở notebook 05. Nếu chuyển máy,
có thể giải nén proof cũ và đặt BASE_RUN_DIR trỏ tới `inputs/run_01`.
Dùng kernel `D:\UIT\.venvs\ml-gpu\Scripts\python.exe`, giữ nguyên phiên bản
thư viện của lượt 05. Môi trường, dữ liệu hay recipe khác sẽ báo lỗi trước training.


In [2]:
BASE_RUN_DIR = ROOT / "results/runs/gru_author_fixed_full_20261027_123100_deeb80e4"
DATA_PATH = ROOT / "data/raw/Base.csv"
NEW_SEEDS = [317, 2026]
ALL_SEEDS = [1234, 317, 2026]

base_report = baf.audit_results([BASE_RUN_DIR], reference="gru_none")
base_manifest = json.loads((BASE_RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
base_config = base_manifest["config"]
baf.require(base_manifest["mode"] == "full", "Base run phải là full.")
baf.require(base_config["seeds"] == [1234], "Base run phải là seed 1234.")
baf.require(
    base_manifest["partitions"]["train"]["n"] == 640000, "Base run không đúng full BAF."
)
baf.require(gru.torch.cuda.is_available(), "Chọn kernel GPU đã chạy notebook 05.")
baf.require(base_manifest["device"] == "cuda", "Base run phải dùng CUDA.")
baf.require(
    base_manifest["python"] == platform.python_version(), "Python khác lượt 05."
)
baf.require(
    base_manifest["packages"] == baf.package_versions(include_torch=True),
    "Phiên bản thư viện khác lượt 05.",
)
baf.require(
    base_manifest["gpu"] == gru.torch.cuda.get_device_name(0), "GPU khác lượt 05."
)
baf.require(
    base_manifest["torch_cuda"] == gru.torch.version.cuda, "CUDA build khác lượt 05."
)
baf.require(
    baf.sha256(DATA_PATH) == base_manifest["data_sha256"], "Base.csv khác lượt 05."
)
baf.require(
    baf._training_signature(BASE_RUN_DIR / "evidence/src")
    == baf._training_signature(ROOT / "src"),
    "Source huấn luyện khác ngoài sửa copy mảng nhãn.",
)
CONFIG = gru.Config(
    **{
        key: value
        for key, value in base_config.items()
        if key in gru.Config.__dataclass_fields__
    }
)
CONFIG.seeds = NEW_SEEDS
CONFIG.validate()
RUN_DIR = baf.new_directory(ROOT / "results/runs", "gru_author_fixed_full_extra_seeds")
print(CONFIG)
print("Giữ seed:", base_config["seeds"])
print("Chạy thêm:", NEW_SEEDS)
print("Output mới:", RUN_DIR)
display(base_report.metrics)


Config(protocol='author_fixed', mode='full', methods=['none', 'balanced'], seeds=[317, 2026], threads=4, device='auto', embedding_dim=186, hidden_dim=733, learning_rate=0.000448103301026688, epochs=8, batch_size=256, dropout=0.3, pilot_train=8192, pilot_eval=4096, save_models=True, dataset_tag='BAF_Base')
Giữ seed: [1234]
Chạy thêm: [317, 2026]
Output mới: D:\UIT\DS317\BAF_project\results\runs\gru_author_fixed_full_extra_seeds_20261027_182144_4cb8cfb7


,protocol,mode,method,seed,partition,threshold,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,initial_model_sha256,...,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp,source_run
0,author_fixed,full,gru_none,1234,validation,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054988,0.027762,0.690141,0.984155,0.989138,150328,7907,874,891,gru_author_fixed_full_20261027_123100_deeb80e4
1,author_fixed,full,gru_none,1234,random_test,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.055045,0.024479,0.692308,0.984081,0.989120,187922,9872,1069,1137,gru_author_fixed_full_20261027_123100_deeb80e4
2,author_fixed,full,gru_balanced,1234,validation,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054825,0.822096,0.039559,0.864960,0.777863,150349,7886,879,886,gru_author_fixed_full_20261027_123100_deeb80e4
3,author_fixed,full,gru_balanced,1234,random_test,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054810,0.828196,0.039941,0.865382,0.778530,187975,9819,1063,1143,gru_author_fixed_full_20261027_123100_deeb80e4


## 2. Train thêm hai seed

Sửa nhãn bằng `to_numpy(dtype=np.int64, copy=True)` để tensor sở hữu mảng ghi được.
Giá trị, dtype, split, model, loss, optimizer và thứ tự minibatch được giữ nguyên.
Không chạy lại seed 1234. Log được ghi mỗi epoch; checkpoint cuối mỗi loss.
Chưa có resume giữa epoch; giữ máy thức và kernel hoạt động.


In [3]:
new_metrics = gru.train(
    data_path=DATA_PATH,
    output=RUN_DIR,
    config=CONFIG,
    expected_rows=1_000_000,
)
display(new_metrics)


{
  "protocol": "author_fixed",
  "mode": "full",
  "methods": [
    "none",
    "balanced"
  ],
  "seeds": [
    317,
    2026
  ],
  "threads": 4,
  "device": "auto",
  "embedding_dim": 186,
  "hidden_dim": 733,
  "learning_rate": 0.000448103301026688,
  "epochs": 8,
  "batch_size": 256,
  "dropout": 0.3,
  "pilot_train": 8192,
  "pilot_eval": 4096,
  "save_models": true,
  "dataset_tag": "BAF_Base",
  "evaluation_partitions": [
    "validation",
    "random_test"
  ]
}
                  n positive                    months
train        640000     7058  [0, 1, 2, 3, 4, 5, 6, 7]
validation   160000     1765  [0, 1, 2, 3, 4, 5, 6, 7]
random_test  200000     2206  [0, 1, 2, 3, 4, 5, 6, 7]
Author fixed recipe: random64/16/20, preprocessing fit outer80% as code. Inner validation features enter preprocessing. No HPO. Seed reset and paired unweighted ablation added. Random split includes month7.
EPOCH {"method": "none", "seed": 317, "epoch": 1, "loss_batch_mean": 0.05101911679091863, "secon

,protocol,mode,method,seed,partition,threshold,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,initial_model_sha256,...,precision_frozen_threshold,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp
0,author_fixed,full,gru_none,317,validation,0.038882,668.466502,1937.917969,405.203125,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.103683,0.055156,0.022663,0.645161,0.984022,0.989081,150325,7910,850,915
1,author_fixed,full,gru_none,317,random_test,0.038882,668.466502,1937.917969,405.203125,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.103433,0.055350,0.018132,0.519481,0.983878,0.988985,187869,9925,1061,1145
2,author_fixed,full,gru_none,2026,validation,0.036337,678.682300,1809.167969,405.024902,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.100614,0.054975,0.017564,0.543860,0.983874,0.989000,150324,7911,880,885
3,author_fixed,full,gru_none,2026,random_test,0.036337,678.682300,1809.167969,405.024902,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.107520,0.054455,0.019492,0.661538,0.983955,0.989075,188074,9720,1035,1171
4,author_fixed,full,gru_balanced,317,validation,0.749175,679.851475,1804.980469,405.025391,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.101651,0.054906,0.734278,0.055411,0.914428,0.858988,150343,7892,872,893
5,author_fixed,full,gru_balanced,317,random_test,0.749175,679.851475,1804.980469,405.025391,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.103417,0.054440,0.735267,0.055381,0.914291,0.858750,188032,9762,1080,1126
6,author_fixed,full,gru_balanced,2026,validation,0.798278,682.192309,1805.839844,405.781738,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.098674,0.054663,0.743909,0.052060,0.907823,0.847750,150352,7883,902,863
7,author_fixed,full,gru_balanced,2026,random_test,0.798278,682.192309,1805.839844,405.781738,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.103847,0.054455,0.752493,0.052500,0.907664,0.847475,188034,9760,1075,1131


## 3. Kiểm chứng và tổng hợp mean ± sample std

Ghép có kiểm tra chặt: cùng dữ liệu, row IDs, preprocessing, packages, GPU,
config (chỉ khác danh sách seed); source chỉ cho phép sửa copy nhãn và audit/export.
Không trộn seed lặp, seed thiếu hoặc run bị ngắt.

Bảng operating points báo riêng ngưỡng 0,5 và ngưỡng FPR5% chọn trên validation,
giữ nguyên khi đánh giá random test. FPR test có thể vượt 5%.
Cột std dùng đơn vị gốc 0–1 cho recall/precision/FPR, không phải điểm phần trăm.
Std giữa 3 seed không phải khoảng tin cậy.


In [4]:
RUN_PATHS = [BASE_RUN_DIR, RUN_DIR]
report = baf.audit_results(
    RUN_PATHS,
    reference="gru_none",
    allow_seed_extension=True,
)
baf.require(set(report.metrics.seed) == set(ALL_SEEDS), "Chưa đủ 3 seed.")
baf.require(len(report.prediction_audit) == 12, "Phải đủ 12 file predictions.")
baf.require(
    (report.metrics.groupby(["method", "partition"]).seed.nunique() == 3).all(),
    "Mỗi nhóm phải có 3 seed.",
)
operating_points = baf.summarize_operating_points(report.metrics)
display(report.metrics)
display(operating_points)
display(report.deltas.rename(columns={"delta_recall_pp": "delta_curve_recall_pp"}))
print("Deltas ở trên dùng ROC curve của từng tập; không dùng để chọn ngưỡng test.")
print(
    "Tổng fit mới (phút):",
    new_metrics.drop_duplicates(["method", "seed"]).fit_seconds.sum() / 60,
)
print("GRU_THREE_SEEDS_READY:", ALL_SEEDS)


,protocol,mode,method,seed,partition,threshold,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,initial_model_sha256,...,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp,source_run
0,author_fixed,full,gru_none,1234,validation,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054988,0.027762,0.690141,0.984155,0.989138,150328,7907,874,891,gru_author_fixed_full_20261027_123100_deeb80e4
1,author_fixed,full,gru_none,1234,random_test,0.059153,676.940642,1929.324219,405.203125,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.055045,0.024479,0.692308,0.984081,0.989120,187922,9872,1069,1137,gru_author_fixed_full_20261027_123100_deeb80e4
2,author_fixed,full,gru_balanced,1234,validation,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054825,0.822096,0.039559,0.864960,0.777863,150349,7886,879,886,gru_author_fixed_full_20261027_123100_deeb80e4
3,author_fixed,full,gru_balanced,1234,random_test,0.851337,673.456177,1940.730469,405.025391,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.054810,0.828196,0.039941,0.865382,0.778530,187975,9819,1063,1143,gru_author_fixed_full_20261027_123100_deeb80e4
4,author_fixed,full,gru_none,317,validation,0.038882,668.466502,1937.917969,405.203125,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.055156,0.022663,0.645161,0.984022,0.989081,150325,7910,850,915,gru_author_fixed_full_extra_seeds_20261027_182...
5,author_fixed,full,gru_none,317,random_test,0.038882,668.466502,1937.917969,405.203125,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.055350,0.018132,0.519481,0.983878,0.988985,187869,9925,1061,1145,gru_author_fixed_full_extra_seeds_20261027_182...
6,author_fixed,full,gru_none,2026,validation,0.036337,678.682300,1809.167969,405.024902,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.054975,0.017564,0.543860,0.983874,0.989000,150324,7911,880,885,gru_author_fixed_full_extra_seeds_20261027_182...
7,author_fixed,full,gru_none,2026,random_test,0.036337,678.682300,1809.167969,405.024902,2dbaac4fb903b1808efda0b4962c137ac663907dfb958e...,...,0.054455,0.019492,0.661538,0.983955,0.989075,188074,9720,1035,1171,gru_author_fixed_full_extra_seeds_20261027_182...
8,author_fixed,full,gru_balanced,317,validation,0.749175,679.851475,1804.980469,405.025391,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.054906,0.734278,0.055411,0.914428,0.858988,150343,7892,872,893,gru_author_fixed_full_extra_seeds_20261027_182...
9,author_fixed,full,gru_balanced,317,random_test,0.749175,679.851475,1804.980469,405.025391,e2a7107879636ed6b2fde6472e5e3c9d3e417017e7c3e9...,...,0.054440,0.735267,0.055381,0.914291,0.858750,188032,9762,1080,1126,gru_author_fixed_full_extra_seeds_20261027_182...


,method,partition,ap_mean,ap_std,roc_auc_mean,roc_auc_std,recall_at_0p5_mean,recall_at_0p5_std,precision_at_0p5_mean,precision_at_0p5_std,recall_frozen_threshold_mean,recall_frozen_threshold_std,fpr_frozen_threshold_mean,fpr_frozen_threshold_std,precision_frozen_threshold_mean,precision_frozen_threshold_std,fit_seconds_mean,fit_seconds_std,n_seeds
0,gru_balanced,random_test,0.155554,0.000883,0.884232,0.000936,0.771985,0.049436,0.049274,0.008210,0.513750,0.003961,0.049447,0.000169,0.103844,0.000426,678.499987,4.522155,3
1,gru_balanced,validation,0.154098,0.004248,0.877640,0.000942,0.766761,0.048163,0.049010,0.008354,0.498961,0.008892,0.049844,0.000029,0.100442,0.001566,678.499987,4.522155,3
2,gru_none,random_test,0.160775,0.003825,0.886356,0.001082,0.020701,0.003341,0.624442,0.092192,0.521759,0.008058,0.049744,0.000538,0.104744,0.002405,674.696481,5.465147,3
3,gru_none,validation,0.151085,0.003699,0.881994,0.000452,0.022663,0.005099,0.626387,0.074926,0.508215,0.008994,0.049985,0.000013,0.101857,0.001615,674.696481,5.465147,3


,reference,method,seed,partition,delta_curve_recall_pp,delta_ap
0,gru_none,gru_balanced,1234,validation,-0.283286,0.002981
1,gru_none,gru_balanced,1234,random_test,0.317316,-0.006948
2,gru_none,gru_balanced,317,validation,-1.246459,0.003680
3,gru_none,gru_balanced,317,random_test,-0.407978,-0.001822
4,gru_none,gru_balanced,2026,validation,-1.246459,0.002378
5,gru_none,gru_balanced,2026,random_test,-1.858568,-0.006892


Deltas ở trên dùng ROC curve của từng tập; không dùng để chọn ngưỡng test.
Tổng fit mới (phút): 45.15320975833333
GRU_THREE_SEEDS_READY: [1234, 317, 2026]


## 4. Xuất proof một lần

ZIP gồm cả seed cũ và hai seed mới, source snapshot tương ứng của từng run,
checkpoint, split/preprocessing, predictions, metric, epoch history và SHA-256.
`operating_points.csv` chứa mean/std; `verify.ipynb` có output kiểm chứng đọc lại.
Không cần Ctrl+S trước export; lưu notebook 06 sau khi xong để giữ output giao diện.


In [5]:
archive = baf.export_proof(
    run_paths=RUN_PATHS,
    output_root=ROOT / "results/proofs",
    reference="gru_none",
    allow_seed_extension=True,
)
print("Proof đủ 3 seed:", archive)


Verified prediction files: 12
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_190838_ef4e3d03.zip
Proof đủ 3 seed: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_190838_ef4e3d03.zip
